# Análise Exploratória de Dados (EDA) — Risco de Diabetes (PNS 2019)

**Projeto Glicuidado — A3 Inteligência Artificial (USJT, 1º Semestre 2026)**

Dataset derivado dos **microdados reais da Pesquisa Nacional de Saúde (PNS) 2019**
(IBGE / Ministério da Saúde), processados por `models/preparar_dados_pns.py`.

**Objetivo:** entender os dados antes de modelar o risco de diabetes a partir de
variáveis de **perfil e estilo de vida** (sem exames laboratoriais).

**Variáveis:** `idade`, `sexo` (1=Masc., 0=Fem.), `imc`, `hipertensao`,
`atividade_fisica`, `tabagismo` (binárias 0/1, exceto idade e imc) e o alvo `diabetes`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

CSV_PATH = os.path.join("..", "data", "diabetes.csv")
df = pd.read_csv(CSV_PATH)
print("Dimensões:", df.shape)
df.head()

In [ ]:
# Tipos, ausentes e estatísticas descritivas
df.info()
print("\nAusentes por coluna:\n", df.isna().sum())
df.describe().T

## 1. Distribuição do alvo

O diabetes é relativamente raro na população, então o conjunto é **desbalanceado**.
Isso justifica (a) avaliar o modelo com **recall, precision, F1 e ROC-AUC** — não só
acurácia — e (b) usar `class_weight="balanced"` na modelagem.

In [ ]:
dist = df["diabetes"].value_counts(normalize=True).rename({0: "Sem diabetes", 1: "Com diabetes"})
print((dist * 100).round(1).astype(str) + " %")
ax = df["diabetes"].map({0: "Sem diabetes", 1: "Com diabetes"}).value_counts().plot(
    kind="bar", color=["#2F6FED", "#E0444B"], rot=0
)
ax.set_title("Distribuição do alvo (diabetes)")
ax.set_ylabel("Nº de pessoas")
plt.tight_layout(); plt.show()

## 2. Idade e IMC por classe

Comparação das distribuições de idade e IMC entre quem tem e não tem diabetes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["idade", "imc"]):
    for valor, rotulo, cor in [(0, "Sem diabetes", "#2F6FED"), (1, "Com diabetes", "#E0444B")]:
        sns.kdeplot(df[df["diabetes"] == valor][col], ax=ax, label=rotulo, color=cor, fill=True, alpha=0.3)
    ax.set_title(f"Distribuição de {col} por classe")
    ax.legend()
plt.tight_layout(); plt.show()

## 3. Prevalência de diabetes por fator de risco

Para as variáveis binárias, a média do alvo dentro de cada grupo é a **prevalência**
de diabetes naquele grupo (em %).

In [ ]:
categoricas = {
    "sexo": {0: "Feminino", 1: "Masculino"},
    "hipertensao": {0: "Não", 1: "Sim"},
    "atividade_fisica": {0: "Não", 1: "Sim"},
    "tabagismo": {0: "Não", 1: "Sim"},
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (col, rotulos) in zip(axes, categoricas.items()):
    prev = df.groupby(col)["diabetes"].mean().rename(index=rotulos) * 100
    prev.plot(kind="bar", ax=ax, color="#2F6FED", rot=0)
    ax.set_title(f"Prevalência por {col}")
    ax.set_ylabel("% com diabetes")
plt.tight_layout(); plt.show()

## 4. Correlação entre variáveis

Idade, hipertensão e IMC tendem a ser as variáveis mais associadas ao diagnóstico —
coerente com a literatura sobre diabetes tipo 2.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlação")
plt.tight_layout(); plt.show()

df.corr(numeric_only=True)["diabetes"].drop("diabetes").sort_values(ascending=False)

## Conclusões da EDA

- **Alvo desbalanceado** (~9% com diabetes) → avaliar com recall/F1/ROC-AUC e usar
  `class_weight="balanced"` no treino.
- **Idade, hipertensão e IMC** são os fatores mais associados ao diagnóstico —
  confirmado depois pela importância das features (permutação e **SHAP**).
- Sem valores ausentes relevantes após o processamento da PNS.
- Variáveis escolhidas por serem **coletáveis sem exames laboratoriais**, viabilizando
  uma ferramenta de **triagem** de baixo custo, aplicável à realidade brasileira.

As decisões de pré-processamento estão em `utils/preprocessamento.py` e a modelagem em
`models/treino_modelo.py`. A explicabilidade por SHAP está em `models/explicar_shap.py`.